# Avance 2. Ingeniería de características

-------
**Equipo 61**

Gustavo Adolfo Morales García A00828432 

Alejandro Jesús Mondragón Jiménez A01795837

Sebastián Ezequiel Coronado Rivera A01212824 

------------

En el avance anterior se realizó el análisis exploratorio del dataset astronómico, identificando problemas relacionados con valores faltantes, asimetrías, diferencias de escala y posibles variables redundantes.

En esta etapa se aplican técnicas de ingeniería de características y preparación de datos para transformar el dataset en una representación más adecuada para modelos supervisados de aprendizaje automático.

In [58]:
# Importación de librerías necesarias
import pandas as pd  # Manipulación y análisis de datos
import numpy as np  # Operaciones numéricas y arrays
import matplotlib.pyplot as plt  # Visualización básica
import seaborn as sns  # Visualización estadística avanzada
from scipy import stats  # Funciones estadísticas
from scipy.stats import skew, kurtosis  # Métricas de distribución
import warnings  # Manejo de advertencias
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer

# Configuración de estilos y opciones
warnings.filterwarnings('ignore')  # Ignorar advertencias para limpieza visual
sns.set_style('whitegrid')  # Estilo de gráficas con cuadrícula blanca
plt.rcParams['figure.figsize'] = (12, 6)  # Tamaño por defecto de figuras
plt.rcParams['font.size'] = 10  # Tamaño de fuente
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.precision', 4)  # Precisión decimal en display

In [59]:
# Carga del archivo CSV inferencia.csv
df = pd.read_csv('inferencia.csv')

df.head()

,name,objra,objdec,C,A,S,nsa_sersic_mass,LogMass,nsa_sersic_ba,nsa_sersic_n,PETRO_TH90,log_age_mean_LW,log_ZH_mean_LW,log_SFR_ssp,log_SFR_Ha,vel_sigma_Re,modelMag_r
0,manga-10001-12701,133.3711,57.5984,2.419,0.191,0.26,3.0680e+09,9.4869,0.3353,0.7418,7.8809,8.5793,-0.6862,0.2797,-0.0895,0.7871,16.3824
1,manga-10001-12702,133.6857,57.4803,2.882,0.094,0.01,5.3416e+09,9.7277,0.5082,1.4427,14.1474,8.6074,-0.5342,0.0641,-0.6085,0.7881,16.6850
2,manga-10001-12703,136.0172,57.0923,3.249,0.135,0.43,1.3694e+10,10.1365,0.2057,2.1808,13.0018,8.7531,-0.3977,0.2267,0.1004,0.5039,15.6903
3,manga-10001-12704,133.9900,57.6780,3.380,0.212,0.85,4.2866e+09,9.6321,0.1500,0.8693,28.6829,8.7495,-0.5560,-0.3412,-0.4651,0.5194,14.6878
4,manga-10001-12705,136.7514,57.4514,2.883,0.152,0.15,1.2987e+10,10.1135,0.4715,1.2505,11.0396,8.5714,-0.6755,0.4678,0.4825,0.7696,15.7142


-------

En el analísis exploratorio de datos se tuvieron multiples hallazgos de los cuales nos basaremos para transformar este dataset en algo que nos ayude a generar un modelo mas eficiente.

## Limpieza de valores placeholder

Durante el análisis exploratorio se identificó la presencia de valores centinela utilizados para representar datos faltantes o mediciones inválidas.

Valores como:
- -999
- -9999
- -102.97
- -118.68

no representan observaciones físicas reales y generan distorsiones importantes en:
- estadísticas descriptivas,
- análisis de distribución,
- detección de outliers,
- y entrenamiento de modelos.

Por esta razón, dichos valores fueron reemplazados por NaN para permitir su tratamiento adecuado durante la etapa de imputación.

In [60]:
#Crear copia del dataset original
df_clean = df.copy()

#REEMPLAZAR PLACEHOLDERS POR NaN

placeholder_map = {
    "C": [-999],
    "A": [-999],
    "S": [-999, -102.97, -118.68],
    "nsa_sersic_mass": [-9999],
    "nsa_sersic_ba": [-9999],
    "nsa_sersic_n": [-9999]
    }

for col, values in placeholder_map.items():
    df_clean[col] = df_clean[col].replace(values, np.nan)

## Imputación de valores faltantes

En el análisis exploratorio se identificó que el porcentaje de valores faltantes era relativamente bajo en la mayoría de las variables (< 2%).

Para evitar pérdida innecesaria de observaciones, se aplicó imputación mediante mediana sobre variables numéricas seleccionadas.

La mediana fue seleccionada debido a su robustez frente a valores atípicos y distribuciones asimétricas presentes en variables astronómicas.

Las variables afectadas incluyen:
- log_SFR_Ha
- log_SFR_ssp
- vel_sigma_Re
- log_age_mean_LW
- log_ZH_mean_LW
- PETRO_TH90
- LogMass
- modelMag_r
- nsa_sersic_mass
- nsa_sersic_ba
- nsa_sersic_n

In [61]:
# Variables seleccionadas para imputación
imputation_cols = [
    "log_SFR_Ha",
    "log_SFR_ssp",
    "vel_sigma_Re",
    "log_age_mean_LW",
    "log_ZH_mean_LW",
    "PETRO_TH90",
    "LogMass",
    "modelMag_r",
    "nsa_sersic_mass",
    "nsa_sersic_ba",
    "nsa_sersic_n"
]

# Filtrar columnas existentes
imputation_cols = [
    col for col in imputation_cols
    if col in df_clean.columns
]

median_imputer = SimpleImputer(strategy="median")

# Aplicar imputación
df_clean[imputation_cols] = median_imputer.fit_transform(
    df_clean[imputation_cols]
)

# Revisar valores faltantes después de imputación
print("\nValores faltantes después de imputación:")
print(df_clean[imputation_cols].isnull().sum())


Valores faltantes después de imputación:
log_SFR_Ha         0
log_SFR_ssp        0
vel_sigma_Re       0
log_age_mean_LW    0
log_ZH_mean_LW     0
PETRO_TH90         0
LogMass            0
modelMag_r         0
nsa_sersic_mass    0
nsa_sersic_ba      0
nsa_sersic_n       0
dtype: int64


## Transformación para variables sesgadas

Se identificaron variables con asimetría fuerte, tanto positiva como negativa. Estas distribuciones pueden afectar la estabilidad de algunos algoritmos de aprendizaje automático y dificultar la convergencia de modelos sensibles a escala y distribución.

Aunque para variables con asimetría positiva se podrían aplicar transformaciones logarítmicas o Box-Cox, y para variables con asimetría negativa se podría utilizar reflexión seguida de logaritmo, se eligió Yeo-Johnson porque permite tratar en un mismo procedimiento variables con valores positivos, negativos y cero.

Las variables transformadas fueron:

- nsa_sersic_mass (79.50)
- PETRO_TH90 (4.03)
- modelMag_r (1.04)
- C (3.24)
- A (-22.66)
- S (-26.97)

La transformación Yeo-Johnson busca reducir la asimetría, estabilizar la varianza y generar distribuciones más adecuadas para modelos de aprendizaje automático.

In [62]:
# Variables identificadas con fuerte asimetría
skewed_cols = [
    "nsa_sersic_mass",
    "PETRO_TH90",
    "modelMag_r",
    "C",
    "A",
    "S"
]

# Filtrar únicamente columnas existentes
skewed_cols = [col for col in skewed_cols if col in df_clean.columns]

yeojohnson_transformer = PowerTransformer(
    method="yeo-johnson",
    standardize=False
)

# Aplicar transformación
df_clean[skewed_cols] = yeojohnson_transformer.fit_transform(
    df_clean[skewed_cols]
)

# Ver asimetría después de la transformación
print("\nAsimetría después de Yeo-Johnson:")
print(df_clean[skewed_cols].skew().sort_values(ascending=False))


Asimetría después de Yeo-Johnson:
S                  2.7336
A                  1.2742
C                  0.2543
nsa_sersic_mass   -0.0183
PETRO_TH90        -0.0315
modelMag_r        -0.0527
dtype: float64


## Estandarización de variables numéricas

Las variables astronómicas presentan escalas significativamente diferentes, lo cual puede afectar la estabilidad y convergencia de modelos de aprendizaje automático.

Por esta razón se aplicó estandarización utilizando StandardScaler.

In [63]:
# Variables a escalar
scaled_cols = [
    "nsa_sersic_mass",
    "PETRO_TH90",
    "modelMag_r",
    "C",
    "A",
    "S",
    "LogMass",
    "nsa_sersic_ba",
    "nsa_sersic_n",
    "log_age_mean_LW",
    "log_ZH_mean_LW",
    "log_SFR_ssp",
    "log_SFR_Ha",
    "vel_sigma_Re",
    "objra",
    "objdec"
]

# Filtrar columnas existentes
scaled_cols = [
    col for col in scaled_cols
    if col in df_clean.columns
]

# Crear scaler
scaler = StandardScaler()

# Aplicar escalamiento
df_clean[scaled_cols] = scaler.fit_transform(
    df_clean[scaled_cols]
)

# Visualizar resultado
df_clean[scaled_cols].head()

,nsa_sersic_mass,PETRO_TH90,modelMag_r,C,A,S,LogMass,nsa_sersic_ba,nsa_sersic_n,log_age_mean_LW,log_ZH_mean_LW,log_SFR_ssp,log_SFR_Ha,vel_sigma_Re,objra,objdec
0,-0.9490,-0.9719,0.8076,-1.3453,-0.0081,-0.0313,-0.9403,-1.2084,-1.2748,-1.1521,-1.3084,0.8322,0.6072,2.0522,-0.6523,1.6126
1,-0.6325,0.1545,1.0683,-0.6738,-0.3510,-0.2956,-0.6049,-0.4241,-0.9179,-1.0850,-0.7010,0.5290,0.1930,2.0572,-0.6482,1.6060
2,-0.0753,-0.0049,0.1766,-0.0947,-0.2084,0.1647,-0.0354,-1.7963,-0.5420,-0.7376,-0.1556,0.7577,0.7587,0.6348,-0.6176,1.5844
3,-0.7591,1.4307,-0.8335,0.1218,0.0685,0.7003,-0.7380,-2.0492,-1.2099,-0.7461,-0.7883,-0.0408,0.3075,0.7127,-0.6442,1.6170
4,-0.1074,-0.3170,0.1993,-0.6722,-0.1483,-0.1513,-0.0674,-0.5909,-1.0158,-1.1710,-1.2657,1.0966,1.0637,1.9645,-0.6080,1.6044


In [65]:
print("DATASET CON TRANSFORMACIONES")
df_clean.head()

DATASET CON TRANSFORMACIONES


,name,objra,objdec,C,A,S,nsa_sersic_mass,LogMass,nsa_sersic_ba,nsa_sersic_n,PETRO_TH90,log_age_mean_LW,log_ZH_mean_LW,log_SFR_ssp,log_SFR_Ha,vel_sigma_Re,modelMag_r
0,manga-10001-12701,-0.6523,1.6126,-1.3453,-0.0081,-0.0313,-0.9490,-0.9403,-1.2084,-1.2748,-0.9719,-1.1521,-1.3084,0.8322,0.6072,2.0522,0.8076
1,manga-10001-12702,-0.6482,1.6060,-0.6738,-0.3510,-0.2956,-0.6325,-0.6049,-0.4241,-0.9179,0.1545,-1.0850,-0.7010,0.5290,0.1930,2.0572,1.0683
2,manga-10001-12703,-0.6176,1.5844,-0.0947,-0.2084,0.1647,-0.0753,-0.0354,-1.7963,-0.5420,-0.0049,-0.7376,-0.1556,0.7577,0.7587,0.6348,0.1766
3,manga-10001-12704,-0.6442,1.6170,0.1218,0.0685,0.7003,-0.7591,-0.7380,-2.0492,-1.2099,1.4307,-0.7461,-0.7883,-0.0408,0.3075,0.7127,-0.8335
4,manga-10001-12705,-0.6080,1.6044,-0.6722,-0.1483,-0.1513,-0.1074,-0.0674,-0.5909,-1.0158,-0.3170,-1.1710,-1.2657,1.0966,1.0637,1.9645,0.1993


En el avance anterior, estas transformaciones se implementaron mediante pipelines y ColumnTransformer para garantizar modularidad y reproducibilidad del preprocesamiento.

Sin embargo, para esta etapa las transformaciones se presentan de forma secuencial y explícita con el objetivo de justificar individualmente cada decisión de procesamiento y mostrar cómo evoluciona el dataset en cada etapa de preparación de datos.

-----

# Generación de nuevas características